In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv('../data/processed/all_matches.csv')
print(df.shape)
print("Loaded successfully!")

(295258, 22)
Loaded successfully!


In [6]:
def engineer_features(df):
    df = df.copy()
    
    # Drop abandoned matches - no winner means we can't label them
    df = df[df['winner'].notna()].reset_index(drop=True)
    
    # Drop super overs - innings 3 and 4
    df = df[df['inning'] <= 2].reset_index(drop=True)

    # ---- INNINGS 1 FEATURES ----
    # Get final score of innings 1 for each match (this becomes the target for innings 2)
    inn1_totals = (
        df[df['inning'] == 1]
        .groupby('match_id')['runs_so_far']
        .max()
        .rename('inn1_total')
        .reset_index()
    )
    df = df.merge(inn1_totals, on='match_id', how='left')

    # ---- CORE PRESSURE FEATURES ----
    df['balls_remaining'] = 120 - df['legal_ball']
    df['wickets_remaining'] = 10 - df['wickets_so_far']

    # Current run rate — runs scored per over so far
    df['current_run_rate'] = (
        df['runs_so_far'] / (df['legal_ball'] / 6)
    ).replace([float('inf'), -float('inf')], 0).fillna(0)

    # ---- INNINGS 2 CHASE FEATURES ----
    inn2 = df['inning'] == 2

    # Use target from JSON if available, else use inn1_total + 1
    df['chase_target'] = df['target'].fillna(df['inn1_total'] + 1)

    df.loc[inn2, 'runs_required'] = (
        df.loc[inn2, 'chase_target'] - df.loc[inn2, 'runs_so_far']
    ).clip(lower=0)

    df.loc[inn2, 'required_run_rate'] = (
        df.loc[inn2, 'runs_required'] / (df.loc[inn2, 'balls_remaining'] / 6)
    ).replace([float('inf'), -float('inf')], 99).fillna(99)

    df.loc[inn2, 'run_rate_diff'] = (
        df.loc[inn2, 'current_run_rate'] - df.loc[inn2, 'required_run_rate']
    )

    # Fill innings 1 chase columns with 0
    df['runs_required'] = df['runs_required'].fillna(0)
    df['required_run_rate'] = df['required_run_rate'].fillna(0)
    df['run_rate_diff'] = df['run_rate_diff'].fillna(0)

    # ---- MATCH PHASE ----
    df['phase'] = pd.cut(
        df['over'],
        bins=[-1, 5, 14, 19],
        labels=[0, 1, 2]  # 0=powerplay, 1=middle, 2=death
    ).astype(int)

    # ---- TOSS ADVANTAGE ----
    df['toss_advantage'] = (
        df['toss_winner'] == df['batting_team']
    ).astype(int)

    # ---- TARGET VARIABLE ----
    df['batting_team_won'] = (
        df['batting_team'] == df['winner']
    ).astype(int)

    return df

# Run it
df_features = engineer_features(df)
print(f"Rows after cleaning: {df_features.shape[0]}")
print(f"Columns: {df_features.shape[1]}")
print(f"\nTarget variable distribution:")
print(df_features['batting_team_won'].value_counts())

Rows after cleaning: 290278
Columns: 33

Target variable distribution:
batting_team_won
0    148212
1    142066
Name: count, dtype: int64


In [5]:
# Fix match_id type first - this makes groupby much faster
df['match_id'] = df['match_id'].astype(str)

# Check it
print(df['match_id'].dtype)
print(df.shape)

str
(295258, 22)


In [7]:
df_features.to_csv('../data/processed/features.csv', index=False)
print("Saved!")

Saved!


In [8]:
# Check innings 2 features look correct
inn2_sample = df_features[df_features['inning'] == 2][
    ['match_id', 'over', 'legal_ball', 'runs_so_far', 
     'runs_required', 'required_run_rate', 
     'current_run_rate', 'run_rate_diff', 
     'balls_remaining', 'wickets_remaining',
     'batting_team_won']
].head(20)

print(inn2_sample.to_string())

    match_id  over  legal_ball  runs_so_far  runs_required  required_run_rate  current_run_rate  run_rate_diff  balls_remaining  wickets_remaining  batting_team_won
125  1082591     0           1            1          207.0          10.436975          6.000000      -4.436975              119                 10                 0
126  1082591     0           2            1          207.0          10.525424          3.000000      -7.525424              118                 10                 0
127  1082591     0           3            1          207.0          10.615385          2.000000      -8.615385              117                 10                 0
128  1082591     0           4            3          205.0          10.603448          4.500000      -6.103448              116                 10                 0
129  1082591     0           5            7          201.0          10.486957          8.400000      -2.086957              115                 10                 0
130  10825